In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Load Model

In [3]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Betr DFS,player_points,Duncan Robinson,Over,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
1,Betr DFS,player_points,Duncan Robinson,Under,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
2,Betr DFS,player_points,Tobias Harris,Over,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
3,Betr DFS,player_points,Tobias Harris,Under,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
4,Betr DFS,player_points,Cade Cunningham,Over,31.5,-137,2025-11-26,2025-11-26T23:26:26Z


## Top EVs for 2 leg bets

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 61 players...
Processing 56 players...
Generated 1455 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
809,Pelle Larsson,Julius Randle,5.5,20.5,-125,-114,13.33,25.02,0.886,0.732,over,over,1,90.86,0.454,High,High
747,Norman Powell,Derik Queen,19.5,13.5,-120,-120,23.72,17.56,0.705,0.715,over,over,1,48.08,0.240,High,High
15,LaMelo Ball,Davion Mitchell,20.5,7.5,-120,-127,23.66,11.01,0.670,0.702,over,over,0,38.21,0.191,High,High
67,Josh Hart,Andrew Wiggins,13.5,14.5,-120,-110,10.86,17.41,0.662,0.660,under,over,0,28.53,0.143,High,High
1284,Brandin Podziemski,Russell Westbrook,9.5,12.5,-125,-130,12.14,15.15,0.648,0.640,over,over,0,22.06,0.110,High,High
534,Collin Murray-Boyles,Donovan Clingan,5.5,10.5,-130,-139,7.18,12.64,0.631,0.629,over,over,0,16.85,0.084,Low,High
447,Isaiah Jackson,Kyle Kuzma,7.5,12.5,-122,-128,9.23,14.80,0.625,0.627,over,over,0,15.11,0.076,Med,High
1383,Moses Moody,Julian Champagnie,11.5,11.5,-130,-121,13.58,13.71,0.617,0.621,over,over,0,12.63,0.063,High,High
637,Myles Turner,Cedric Coward,12.5,13.5,-130,-122,14.52,15.17,0.617,0.604,over,over,0,9.45,0.047,High,High
606,Bam Adebayo,Keegan Murray,16.5,15.5,-132,-108,18.67,17.43,0.613,0.602,over,over,0,8.48,0.042,High,High


### Prizepicks picks

In [16]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 98 players...
Processing 90 players...
Generated 3784 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
2026,Ryan Rollins,Harrison Barnes,18.5,12.5,-115,-127,23.64,16.68,over,over,0.756,0.738,0.5472,0.221,0.179,0.254,64.17,0.321,1,7.40,6.55,High,High,"(9.1, 38.1)","(3.8, 29.5)",0.05,0,64.2
2440,Julius Randle,Saddiq Bey,20.5,12.5,-114,-143,25.02,16.86,over,over,0.732,0.723,0.5191,0.200,0.135,0.212,55.73,0.279,1,7.28,7.35,High,High,"(10.7, 39.3)","(2.4, 31.3)",0.05,0,55.7
2104,Andrew Wiggins,Jeremiah Fears,13.5,14.5,-145,-135,17.41,18.48,over,over,0.711,0.716,0.4985,0.119,0.141,0.165,49.54,0.248,0,7.05,6.98,High,High,"(3.6, 31.2)","(4.8, 32.2)",0.05,0,49.5
1930,Norman Powell,Derik Queen,19.5,14.0,-120,-137,23.72,17.56,over,over,0.705,0.691,0.4769,0.159,0.113,0.168,43.06,0.215,0,7.84,7.15,High,High,"(8.4, 39.1)","(3.5, 31.6)",0.05,0,43.1
2392,Davion Mitchell,Jock Landale,7.5,9.0,-127,-137,11.01,11.76,over,over,0.702,0.678,0.4660,0.142,0.099,0.150,39.81,0.199,0,6.61,6.00,High,Med,"(0.0, 24.0)","(0.0, 23.5)",0.05,0,39.8
19,LaMelo Ball,Tyler Herro,20.5,19.5,-120,-125,23.66,21.05,over,over,0.670,0.664,0.4361,0.124,0.109,0.139,30.83,0.154,0,7.19,3.66,High,Low,"(9.6, 37.7)","(13.9, 28.2)",0.05,0,30.8
3656,Mark Williams,Jerami Grant,14.5,20.5,-155,-134,12.08,17.80,under,under,0.662,0.660,0.4278,0.054,0.087,0.086,28.33,0.142,0,5.80,6.57,Med,High,"(0.7, 23.4)","(4.9, 30.7)",0.05,0,28.3
1760,Ben Sheppard,Brandin Podziemski,6.5,9.5,-105,-125,8.76,12.14,over,over,0.654,0.648,0.4157,0.142,0.093,0.137,24.71,0.124,0,5.68,6.94,Med,High,"(0.0, 19.9)","(0.0, 25.8)",0.05,0,24.7
951,Bennedict Mathurin,Russell Westbrook,20.5,12.5,-145,-130,23.07,15.15,over,over,0.647,0.640,0.4060,0.055,0.075,0.078,21.79,0.109,0,6.83,7.37,High,High,"(9.7, 36.5)","(0.7, 29.6)",0.05,0,21.8
1824,Collin Murray-Boyles,Aaron Holiday,5.5,10.5,-130,-130,7.18,8.59,over,under,0.631,0.628,0.3885,0.066,0.063,0.075,16.56,0.083,0,5.00,5.85,Low,Med,"(0.0, 17.0)","(0.0, 20.1)",0.05,0,16.6


## 3 leg parlay

### Underdog picks

In [17]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 61 players...
Processing 56 players...
Generated 23276 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
16808,Pelle Larsson,Julius Randle,Derik Queen,5.5,20.5,13.5,13.33,25.02,17.56,0.886,0.732,0.715,over,over,over,1,150.57,0.301,High,High,High
642,LaMelo Ball,Norman Powell,Brandin Podziemski,20.5,19.5,9.5,23.66,23.72,12.14,0.670,0.705,0.648,over,over,over,0,65.24,0.130,High,High,High
2037,Josh Hart,Davion Mitchell,Russell Westbrook,13.5,7.5,12.5,10.86,11.01,15.15,0.662,0.702,0.640,under,over,over,0,60.66,0.121,High,High,High
11328,Collin Murray-Boyles,Andrew Wiggins,Donovan Clingan,5.5,14.5,10.5,7.18,17.41,12.64,0.631,0.660,0.629,over,over,over,0,41.73,0.083,Low,High,High
10255,Isaiah Jackson,Kyle Kuzma,Julian Champagnie,7.5,12.5,11.5,9.23,14.80,13.71,0.625,0.627,0.621,over,over,over,0,31.29,0.063,Med,High,High


### Prizepicks picks

In [19]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 98 players...
Processing 90 players...
Generated 98658 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
67510,Ryan Rollins,Julius Randle,Harrison Barnes,18.5,20.5,12.5,23.64,25.02,16.68,0.756,0.732,0.738,over,over,over,1,120.87,0.242,High,High,High
1604,LaMelo Ball,Andrew Wiggins,Saddiq Bey,20.5,13.5,12.5,23.66,17.41,16.86,0.670,0.711,0.723,over,over,over,0,85.89,0.172,High,High,High
66428,Norman Powell,Jeremiah Fears,Mark Williams,19.5,14.5,14.5,23.72,18.48,12.08,0.705,0.716,0.662,over,over,under,0,80.23,0.160,High,High,Med
78199,Davion Mitchell,Derik Queen,Jerami Grant,7.5,14.0,20.5,11.01,17.56,17.80,0.702,0.691,0.660,over,over,under,0,72.64,0.145,High,High,High
60259,Ben Sheppard,Tyler Herro,Jock Landale,6.5,19.5,9.0,8.76,21.05,11.76,0.654,0.664,0.678,over,over,over,0,59.03,0.118,Med,Low,Med


In [9]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)